# Phase 2: Fine-tune Laya on Jigsaw (Kaggle 2×T4)

Adapted from the official `laya_finetune_typed_decisions_2xT4_kaggle.ipynb`: same training loop, our data.

**Settings (right panel):** Accelerator **GPU T4 ×2** · Internet **On** · Add Input → **jigsaw-toxic-comment-classification-challenge** · Add-ons → Secrets → attach **HF_TOKEN**.

**Two modes (cell below):**
- `RUN_MODE = "smoke"` → ~15–30 min timing test on a tiny slice. Run this first, interactively.
- `RUN_MODE = "full"` → real training. Set it, then **Save Version → Save & Run All** so it runs in the background (you can close the tab).

In [1]:
# ==== CONFIG ====
RUN_MODE = "full"          # "smoke" (timing test) or "full" (real training)
N_TRAIN_COMMENTS = 12000    # full run: training comments (each = 6 questions = 6 training items)
EPOCHS_FULL = 2
N_CALIB_COMMENTS = 1000     # untouched validation comments used only for the temperature fit

SMOKE = RUN_MODE == "smoke"
n_train = 500 if SMOKE else N_TRAIN_COMMENTS
n_calib = 200 if SMOKE else N_CALIB_COMMENTS
epochs = 1 if SMOKE else EPOCHS_FULL
print(f"Mode={RUN_MODE} | train comments={n_train} | calib comments={n_calib} | epochs={epochs}")

Mode=full | train comments=12000 | calib comments=1000 | epochs=2


## 1. GPU check + install

In [2]:
import os, torch
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)")
assert n_gpu >= 2, "Switch Accelerator to GPU T4 x2"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U laya
import laya; print("laya", laya.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

GPU 0: Tesla T4 (15.6 GB)
GPU 1: Tesla T4 (15.6 GB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 3.3 MB/s eta 0:00:00
laya 0.3.20 | torch 2.10.0+cu128 | CUDA True


## 2. Build training data

Same splits as Phase 1 (same seeds), so the test sample is identical and scores are comparable.
Training set is balanced (half toxic, all rare-label comments kept); calibration set is untouched validation data.

In [3]:
import glob, json, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
QUESTIONS = {   # identical to Phase 1 zero-shot, and must stay identical at inference
  'toxic':         {'type':'noul','instructions':'Is this comment rude, disrespectful or unreasonable?'},
  'severe_toxic':  {'type':'noul','instructions':'Is this comment extremely hateful, aggressive or disrespectful?'},
  'obscene':       {'type':'noul','instructions':'Does this comment contain obscene or vulgar language?'},
  'threat':        {'type':'noul','instructions':'Does this comment threaten to harm someone?'},
  'insult':        {'type':'noul','instructions':'Does this comment insult a person?'},
  'identity_hate': {'type':'noul','instructions':'Does this comment attack someone for their identity (race, religion, gender, sexuality, etc.)?'},
}
MAX_CHARS = 1200
SMOOTH = 0.01   # targets 0.99 / 0.01 instead of 1 / 0

# --- data (same steps + seeds as Phase 1) ---
BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv*', recursive=True)[0])
def load(name):
    for ext in ['.csv', '.csv.zip']:
        p = f'{BASE}/{name}{ext}'
        if os.path.exists(p): return pd.read_csv(p)
    raise FileNotFoundError(name)
train = load('train')
test = load('test').merge(load('test_labels'), on='id')
test = test[test['toxic'] != -1].reset_index(drop=True)
train = train.drop_duplicates('comment_text').reset_index(drop=True)
tr, va = train_test_split(train, test_size=0.1, random_state=42, stratify=train['toxic'])
sample = test.sample(10000, random_state=42)

# --- balanced training pick ---
def pick_train(df, n, seed=42):
    any_pos = df[LABELS].max(axis=1) == 1
    rare = df[['severe_toxic','threat','identity_hate']].max(axis=1) == 1
    n_pos = n // 2
    rare_rows = df[rare].sample(min(rare.sum(), n_pos), random_state=seed)
    other = df[any_pos & ~rare]
    other_rows = other.sample(min(len(other), n_pos - len(rare_rows)), random_state=seed)
    pos = pd.concat([rare_rows, other_rows])
    neg = df[~any_pos].sample(n - len(pos), random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed)

train_df = pick_train(tr, n_train)
calib_df = va.sample(n_calib, random_state=7)          # natural distribution, never trained on
print('train label rates:', train_df[LABELS].mean().round(3).to_dict())
print('calib label rates:', calib_df[LABELS].mean().round(3).to_dict())

# --- tokenizer + config from the base English checkpoint (skip the other bundled models) ---
MODEL_ID = "convaiinnovations/laya"
model_dir = snapshot_download(MODEL_ID, ignore_patterns=["multilingual/*", "typed-decisions/*", "eval/*", "assets/*"])
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
cfg = json.load(open(os.path.join(model_dir, "rl_agent_config.json")))
print("max_len", cfg["max_len"], "| head_max_len", cfg["head_max_len"])

def build_items(df):
    items, skipped = [], 0
    for row in df[['comment_text'] + LABELS].itertuples(index=False):
        text, labs = row[0], row[1:]
        state = {'comment': text[:MAX_CHARS]}
        for l, y in zip(LABELS, labs):
            q = QUESTIONS[l]; crit = q.get('criteria', {})
            k = len(render_options({"t": "noul", "crit": crit}))
            seq, markers = build_sequence(tok, state, {"t": "noul", "ins": q["instructions"], "crit": crit},
                                          cfg["max_len"], cfg["head_max_len"])
            if len(markers) != k:
                skipped += 1; continue
            target = [SMOOTH, 1 - SMOOTH] if y == 1 else [1 - SMOOTH, SMOOTH]   # order: [P(false), P(true)]
            items.append({"ids": seq, "markers": markers, "qtype": QTYPES["noul"],
                          "target": target, "label": int(y)})
    return items, skipped

train_items, s1 = build_items(train_df)
calib_items, s2 = build_items(calib_df)
torch.save(train_items, "/kaggle/working/train_items.pt")
torch.save(calib_items, "/kaggle/working/calib_items.pt")
lens = [len(it["ids"]) for it in train_items]
print(f"train items {len(train_items)} (skipped {s1}) | calib items {len(calib_items)} (skipped {s2})")
print(f"sequence length: mean {np.mean(lens):.0f}, max {max(lens)}")

train label rates: {'toxic': 0.472, 'severe_toxic': 0.12, 'obscene': 0.309, 'threat': 0.036, 'insult': 0.293, 'identity_hate': 0.104}
calib label rates: {'toxic': 0.102, 'severe_toxic': 0.009, 'obscene': 0.059, 'threat': 0.007, 'insult': 0.049, 'identity_hate': 0.015}


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

max_len 512 | head_max_len 192
train items 72000 (skipped 0) | calib items 6000 (skipped 0)
sequence length: mean 118, max 512


## 3. Training script

Official RLCD loop, with 4 changes (marked `# CHANGED`):
1. Calibration items come from untouched validation data, not from the (balanced) training set.
2. `max_len` / `head_max_len` are **not** overridden, so training matches inference exactly.
3. Epoch count comes from the config cell.
4. Micro-batch 16 × accumulation 2 (same effective batch of 64 as official; our sequences are shorter), plus an ETA printout.

In [4]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
            "target": target, "qtype": torch.tensor([it["qtype"] for it in items]),
            "label": torch.tensor([it["label"] for it in items])}

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank(); world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    model_dir, output_dir = sys.argv[1], sys.argv[2]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    # CHANGED: no max_len / head_max_len override -> training matches inference

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device); model.train()
    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    # CHANGED: separate files; calibration items are untouched validation data
    train_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)
    calib_items = torch.load("/kaggle/working/calib_items.pt", weights_only=False)
    my_items = train_items[rank::world_size]

    EPOCHS = int(os.environ.get("EPOCHS", "2"))   # CHANGED
    MICRO_BATCH = 16     # CHANGED (official 8; our sequences are short)
    GRAD_ACCUM = 2       # CHANGED (official 4) -> effective batch still 16*2*2 = 64
    GROUP_SIZE = 4
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENCODER},
                                   {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01)
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    steps_per_epoch = math.ceil(len(my_items) / MICRO_BATCH)
    total_steps = steps_per_epoch * EPOCHS
    global_step = 0
    if rank == 0:
        print(f"Training: {len(train_items)} items | {len(my_items)} per GPU | {EPOCHS} epochs | {total_steps} steps per GPU", flush=True)
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank); random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk: continue
            batch = collate_train_batch(chunk, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                                        batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                                        batch["qtype"].to(device))
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)

            # 1. sample G noisy versions of the prediction (exploration)
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            # 2. score each version with proper scoring rules; advantage = better/worse than group average
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            # 3. policy-gradient loss + plain soft cross-entropy
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1; global_step += 1
            if rank == 0 and n_batches % 50 == 0:
                el = time.time() - t0
                eta = el / global_step * (total_steps - global_step)
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches}/{steps_per_epoch} | Loss {loss.item()*GRAD_ACCUM:.4f} "
                      f"| Reward {r.mean().item():.3f} | elapsed {el/60:.1f} min | ETA {eta/60:.1f} min", flush=True)

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} done at {(time.time()-t0)/60:.1f} min | Avg loss {epoch_loss/max(1,n_batches):.4f} ===", flush=True)
        dist.barrier()
        if rank == 0:   # rolling checkpoint so a crash/timeout doesn't lose everything
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest"); os.makedirs(ckpt_dir, exist_ok=True)
            save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()},
                      os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            json.dump({"epoch": epoch + 1, "total_epochs": EPOCHS}, open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w"))

    train_seconds = time.time() - t0
    dist.barrier()

    if rank == 0:
        print("\nFitting calibration temperature on untouched validation items...", flush=True)
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache(); model.eval()
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(cb["input_ids"].to(device), cb["attention_mask"].to(device),
                                     cb["marker_pos"].to(device), cb["marker_mask"].to(device), cb["qtype"].to(device))
                l_np = l_sub.float().cpu().numpy()
                for r_i, it in enumerate(c_chunk):
                    calib_preds.append((it["qtype"], l_np[r_i, :len(it["markers"])], it["target"]))
        fitted = [1.2, 1.2, 1.2]
        for qt in range(3):
            sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
            if sel: fitted[qt] = fit_one_temp(sel)
        print("Fitted temperatures (choice, score, noul):", [round(t, 3) for t in fitted], flush=True)

        os.makedirs(output_dir, exist_ok=True)
        save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()},
                  os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-jigsaw-moderation"
        cfg["temperature"] = fitted
        cfg.pop("temperature_by_options", None)
        json.dump(cfg, open(os.path.join(output_dir, "rl_agent_config.json"), "w"), indent=2)
        json.dump({"train_items": len(train_items), "epochs": EPOCHS, "train_seconds": train_seconds,
                   "items_per_sec": len(train_items) * EPOCHS / train_seconds},
                  open("/kaggle/working/train_timing.json", "w"), indent=2)
        print(f"Saved model to {output_dir} | training took {train_seconds/60:.1f} min", flush=True)

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

Writing /kaggle/working/train_ddp.py


## 4. Train (both GPUs)

In [5]:
OUTPUT_DIR = "/kaggle/working/laya_jigsaw"
!EPOCHS={epochs} torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {model_dir} {OUTPUT_DIR}

timing = json.load(open("/kaggle/working/train_timing.json"))
print(timing)
if SMOKE:
    full_items = N_TRAIN_COMMENTS * len(LABELS) * EPOCHS_FULL
    est_h = full_items / timing["items_per_sec"] / 3600
    print(f"\nProjected FULL run ({N_TRAIN_COMMENTS} comments x {EPOCHS_FULL} epochs): ~{est_h:.1f} hours of training "
          f"(+ ~15 min setup/eval).")

W0924 20:43:14.553000 85 torch/distributed/run.py:852] 
W0924 20:43:14.553000 85 torch/distributed/run.py:852] *****************************************
W0924 20:43:14.553000 85 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0924 20:43:14.553000 85 torch/distributed/run.py:852] *****************************************
[W924 20:43:14.109285293 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W924 20:43:22.827286800 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W924 20:43:22.867531472 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Training: 72000 items | 36000 per GPU | 2 epochs | 4500 steps per GPU
[rank0]:W0924 20:43:50.574000 93 torch/_inductor/utils.py:1679] [1/0_1] Not enough SM

## 5. Evaluate on the same test sample as Phase 1
Smoke mode uses 1,000 comments just to check the pipeline works (score will be meaningless after so little training).

In [6]:
import time
from sklearn.metrics import roc_auc_score
from laya.common import ece_score

agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")
eval_df = sample.iloc[:1000] if SMOKE else sample
states = [{'comment': t[:MAX_CHARS]} for t in eval_df.comment_text.tolist()]
t0 = time.time()
res = agent_ft.predict_batch(states, QUESTIONS, batch_size=16)
t = time.time() - t0
pred = pd.DataFrame([{l: float(r["answers"][l]["noul"]) for l in LABELS} for r in res], index=eval_df.index)
pred.to_csv("/kaggle/working/laya_finetuned_preds.csv")

aucs = {l: roc_auc_score(eval_df[l], pred[l]) for l in LABELS}
aucs["mean"] = float(np.mean(list(aucs.values())))
p = pred.values.ravel(); y = eval_df[LABELS].values.ravel()
ece = ece_score(np.maximum(p, 1 - p), ((p >= 0.5) == (y == 1)).astype(float))

board = pd.DataFrame([
    {"model": "TF-IDF (Phase 1)", "mean_auc": 0.9769},
    {"model": "Laya zero-shot (Phase 1)", "mean_auc": 0.9708},
    {"model": "Detoxify (Phase 1)", "mean_auc": 0.9854},
    {"model": f"Laya fine-tuned ({RUN_MODE})", "mean_auc": round(aucs['mean'], 4)},
])
print(pd.Series(aucs).round(4).to_string()); print()
print(board.to_string(index=False))
print(f"\nECE (calibration error, lower is better): {ece:.4f} | speed {len(states)/t:.1f} comments/sec")
json.dump({"aucs": aucs, "ece": float(ece), "comments_per_sec": len(states)/t, "mode": RUN_MODE},
          open("/kaggle/working/finetune_results.json", "w"), indent=2)

toxic            0.9648
severe_toxic     0.9869
obscene          0.9795
threat           0.9528
insult           0.9811
identity_hate    0.9867
mean             0.9753

                   model  mean_auc
        TF-IDF (Phase 1)    0.9769
Laya zero-shot (Phase 1)    0.9708
      Detoxify (Phase 1)    0.9854
  Laya fine-tuned (full)    0.9753

ECE (calibration error, lower is better): 0.0134 | speed 6.5 comments/sec


## 6. Push to Hugging Face (full mode only, private repo)

In [7]:
if SMOKE:
    print("Smoke mode: skipping upload.")
else:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    repo = f"{api.whoami()['name']}/laya-jigsaw-moderation"
    card = f"""---
license: apache-2.0
base_model: convaiinnovations/laya
tags: [laya, toxicity, moderation, calibrated-decisions]
---
# Laya fine-tuned for toxic comment moderation (Jigsaw)

Answers 6 yes/no toxicity questions per comment (toxic, severe_toxic, obscene, threat, insult, identity_hate).

| Model | Mean ROC-AUC (10k Jigsaw test sample) |
|---|---|
| **This model** | **{aucs['mean']:.4f}** |
| Detoxify original | 0.9854 |
| TF-IDF + LR | 0.9769 |
| Laya zero-shot | 0.9708 |

ECE: {ece:.4f}. Trained on {N_TRAIN_COMMENTS} balanced comments x {EPOCHS_FULL} epochs on Kaggle 2xT4.
"""
    open(os.path.join(OUTPUT_DIR, "README.md"), "w").write(card)
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=repo, ignore_patterns=["checkpoint_latest/*"],
                      commit_message=f"Jigsaw fine-tune: mean AUC {aucs['mean']:.4f}")
    print(f"Uploaded (private): https://huggingface.co/{repo}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded (private): https://huggingface.co/Vinay57/laya-jigsaw-moderation
